# 06.1 · Unsupervised learning: what you can do without a target

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/01-what-is-unsupervised/lesson.ipynb)

Read the [README](README.md) first. This notebook runs one small example of each kind of unsupervised learning on real ShopWave customers. **Do not worry about how each method works**; every one gets its own lesson. Watch for one thing: every `fit` call takes `X` only, never `y`.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

## 1. The RFM table

One row per customer, three numbers:

- **recency**: days since their last order (as of 10 December 2011),
- **frequency**: how many separate orders they placed,
- **monetary**: how much they spent in total, in pounds.

There is no target column. Nobody has told us which customers are "good".

In [2]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)    # 2011-12-10
rfm = sales.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum"))
print(f"{len(sales):,} rows, {len(rfm):,} customers, snapshot {snapshot.date()}")

392,693 rows, 4,338 customers, snapshot 2011-12-10


In [3]:
rfm.describe().round(1)

,recency,frequency,monetary
count,4338.0,4338.0,4338.0
mean,92.1,4.3,2048.7
std,100.0,7.7,8985.2
min,0.0,1.0,3.8
25%,17.0,1.0,306.5
50%,50.0,2.0,668.6
75%,141.8,5.0,1660.6
max,373.0,209.0,280206.0


All three columns have long right tails: the median customer spent £669, the biggest £280,206. Distance-based methods need comparable scales, so we take logs and standardise (you met both in 04.5 and 04.6):

In [4]:
from sklearn.preprocessing import StandardScaler

rfm_log = np.log1p(rfm)                         # tame the long right tails
X = StandardScaler().fit_transform(rfm_log)      # mean 0, std 1 per column
print("X shape:", X.shape)

X shape: (4338, 3)


## 2. Clustering: which customers are alike?

K-Means with 4 groups. Note the call: `fit_predict(X)`, no answers given.

In [5]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=42)
rfm["segment"] = km.fit_predict(X)

profile = rfm.groupby("segment").agg(customers=("recency", "size"), recency=("recency", "mean"),
                                     frequency=("frequency", "mean"), monetary=("monetary", "mean"))
profile["share_of_revenue"] = rfm.groupby("segment")["monetary"].sum() / rfm["monetary"].sum()
profile.sort_values("monetary", ascending=False).round(2)

,customers,recency,frequency,monetary,share_of_revenue
segment,,,,,
0,699,11.37,13.88,8194.29,0.64
3,1178,69.95,4.14,1814.50,0.24
2,826,17.28,2.18,555.82,0.05
1,1635,180.27,1.32,344.24,0.06


Read the table top to bottom. The first group bought recently, often and a lot, and brings in about two thirds of all revenue. Another large group bought once, long ago, for little. Nobody told the algorithm about "loyal" or "lapsed"; it found customers who sit close together in R-F-M space. Naming and using these groups is lesson 06.3.

## 3. Oddities: which customers are unlike the rest?

The simplest anomaly check: how many standard deviations is each value from the mean (the z-score from 01.8)?

In [6]:
rfm_raw = rfm[["recency", "frequency", "monetary"]]
z = (rfm_raw - rfm_raw.mean()) / rfm_raw.std()
odd = (z.abs() > 3).any(axis=1)
print(f"{odd.sum()} customers are more than 3 standard deviations from the mean on R, F or M")
rfm_raw[odd].sort_values("monetary", ascending=False).head(5)

64 customers are more than 3 standard deviations from the mean on R, F or M


,recency,frequency,monetary
Customer ID,,,
14646.0,1,73,280206.02
18102.0,0,60,259657.30
17450.0,8,46,194390.79
16446.0,0,2,168472.50
14911.0,1,201,143711.17


These are not necessarily fraud. They are the customers worth a human look: some will be wholesalers, some data errors, perhaps one or two something worse. Lesson 06.8 does this properly.

## 4. Compression: can fewer numbers describe each customer?

PCA looks for new directions through the data that carry as much variation as possible. How much does each direction carry?

In [7]:
from sklearn.decomposition import PCA

pca = PCA().fit(X)
for i, share in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"direction {i}: {share:.1%} of the variation")

direction 1: 75.2% of the variation
direction 2: 18.7% of the variation
direction 3: 6.1% of the variation


One direction already carries three quarters of the variation: customers who buy often also tend to spend a lot and to have bought recently, so R, F and M largely move together. With thousands of columns (one per product) this kind of compression becomes essential. Lesson 06.6.

## 5. Associations: what is bought together?

Count every pair of products that appears on the same invoice.

In [8]:
from collections import Counter
from itertools import combinations

baskets = sales.groupby("Invoice")["Description"].apply(lambda s: sorted(set(s)))
pair_counts = Counter()
for items in baskets:
    if len(items) <= 60:                         # skip huge wholesale invoices (too many pairs)
        pair_counts.update(combinations(items, 2))
print(f"{len(baskets):,} invoices")

18,532 invoices


In [9]:
for (a, b), n in pair_counts.most_common(3):
    print(f"{n:>4} invoices:  {a.strip()}  +  {b.strip()}")

 462 invoices:  JUMBO BAG PINK POLKADOT  +  JUMBO BAG RED RETROSPOT
 454 invoices:  GREEN REGENCY TEACUP AND SAUCER  +  ROSES REGENCY TEACUP AND SAUCER
 445 invoices:  ALARM CLOCK BAKELIKE GREEN  +  ALARM CLOCK BAKELIKE RED


Raw counts favour popular products. Lesson 06.9 adjusts for that with *lift*: how much more often two products appear together than chance would predict.

## 6. Judging without a target: stability

There is no accuracy to compute. One thing we *can* check: do we get the same groups if we run K-Means again with a different random start? The **adjusted Rand index** (ARI) compares two groupings: 1 means identical, 0 means no more alike than chance.

In [10]:
from sklearn.metrics import adjusted_rand_score

first = KMeans(n_clusters=4, n_init=1, random_state=0).fit_predict(X)
for seed in [1, 2, 3, 4]:
    other = KMeans(n_clusters=4, n_init=1, random_state=seed).fit_predict(X)
    print(f"seed 0 vs seed {seed}: ARI = {adjusted_rand_score(first, other):.3f}")

seed 0 vs seed 1: ARI = 0.970
seed 0 vs seed 2: ARI = 0.970
seed 0 vs seed 3: ARI = 0.986
seed 0 vs seed 4: ARI = 0.965


Very close to 1: the four groups are a stable feature of the data, not an accident of where the algorithm started. That, plus groups the business can describe, is what a good unsupervised result looks like.

## Quiz answers

1. A **target** column ($y$): the known right answer for each row.
2. **Clustering, anomaly detection, dimensionality reduction, association rules, recommendation.**
3. Supervised: `fit(X, y)`. Unsupervised: `fit(X)`.
4. **Tightness** (e.g. silhouette: are points close to their own group?), **stability** (same groups with a different seed or sample?), and **usefulness** (can the business name and act on each group?).
5. **No.** Usefulness is the final test. A segmentation nobody can explain or act on has no value, whatever its score.

**Next:** [06.2 K-Means by hand](../02-kmeans-by-hand/)